In [ ]:
import numpy as np
import tensorflow.keras.backend as K
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Embedding, Lambda
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.preprocessing import sequence
from tensorflow.keras.preprocessing.text import Tokenizer
import gensim
import os

class np_utils:
    to_categorical = staticmethod(to_categorical)

file_path = 'corona.txt' if os.path.exists('corona.txt') else (
    'DL-assignments/corona.txt' if os.path.exists('DL-assignments/corona.txt') else 'D:/corona.txt'
)
data = open(file_path, 'r')
corona_data = [text for text in data if text.count(' ') >= 2]
vectorize = Tokenizer()
vectorize.fit_on_texts(corona_data)
corona_data = vectorize.texts_to_sequences(corona_data)
total_vocab = sum(len(s) for s in corona_data)
word_count = len(vectorize.word_index) + 1
window_size = 2
vocab_size = max(total_vocab, word_count)
print('total_vocab:', total_vocab)
print('word_count:', word_count)

In [ ]:
def cbow_model(data, window_size, total_vocab):
    total_length = window_size*2
    for text in data:
        text_len = len(text)
        for idx, word in enumerate(text):
            context_word = []
            target   = []            
            begin = idx - window_size
            end = idx + window_size + 1
            context_word.append([text[i] for i in range(begin, end) if 0 <= i < text_len and i != idx])
            target.append(word)
            contextual = sequence.pad_sequences(context_word, maxlen=total_length)
            final_target = np_utils.to_categorical(target, total_vocab)
            yield(contextual, final_target)

In [ ]:
model = Sequential()
model.add(Embedding(input_dim=vocab_size, output_dim=100, input_length=window_size*2))
model.add(Lambda(lambda x: K.mean(x, axis=1), output_shape=(100,)))
model.add(Dense(vocab_size, activation='softmax'))
model.compile(loss='categorical_crossentropy', optimizer='adam')
for i in range(10):
    cost = 0
    for x, y in cbow_model(corona_data, window_size, vocab_size):
        cost += model.train_on_batch(x, y)
    print(i, cost)

In [ ]:
dimensions = 100
vect_path = 'vectors.txt' if os.path.exists('.') else 'D:/vectors.txt'
vect_file = open(vect_path, 'w')
vect_file.write('{} {}\n'.format(len(vectorize.word_index), dimensions))

In [ ]:
weights = model.get_weights()[0]
for text, i in vectorize.word_index.items():
    final_vec = ' '.join(map(str, list(weights[i, :])))
    vect_file.write('{} {}\n'.format(text, final_vec))
vect_file.close()

In [ ]:
cbow_output = gensim.models.KeyedVectors.load_word2vec_format(vect_path, binary=False)
cbow_output.most_similar(positive=['the'])

In [ ]:
cbow_output['the'].shape